# 最適作物マップ（65か国・5シナリオ）

- **最適作物**: カロリーベース（kcal/ha）で単収が最も高い穀物（Maize / Rice / Wheat）
- **係数**: 全世界同一（Maize 3960, **Rice 3840**（0.8なし・玄米）、Wheat 3750）
- **5シナリオ**: 2013（2011–2013平均）、2100 SSP370 平均・95%CI下限、2100 SSP245 平均・95%CI下限
- **対象**: 予測精度上位65か国。2013は **regression_df（2011–2013年 平均）**、2100は predict。地図は **ISO-3** で描画し全65か国を表示。

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.express as px

base_dir = r'C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット'
os.chdir(base_dir)

reg = pd.read_excel(os.path.join(base_dir, 'regression_df_20260112_233447.xlsx'))
pred = pd.read_csv(os.path.join(base_dir, 'predict_2100_bootstrap_results.csv'))

ITEMS = ['Maize (corn)', 'Rice', 'Wheat']
KCAL_PER_KG = {'Maize (corn)': 3960, 'Rice': 3072, 'Wheat': 3750}

COUNTRY_65 = {
    'Greece', 'Zambia', 'Italy', 'Thailand', 'Canada', 'Peru', 'United States of America',
    'Dominican Republic', 'Republic of Korea', 'Kazakhstan', 'El Salvador', 'Suriname', 'Eswatini',
    'Oman', 'Austria', 'Kuwait', 'Costa Rica', 'Argentina', 'Namibia', 'Chile', 'Spain', 'Panama',
    'Ecuador', 'Myanmar', 'Belgium', 'Qatar', 'Uruguay', 'Guyana', 'Morocco', 'Honduras', 'Australia',
    'Mauritania', 'Colombia', 'France', 'Japan', 'Haiti', 'New Zealand', 'Fiji', 'Niger', 'Switzerland',
    'Poland', 'Russian Federation', 'Somalia', 'Guatemala', 'Comoros', 'Mexico', 'Portugal', 'Armenia',
    'Türkiye', 'Slovenia', 'China', 'Syrian Arab Republic', 'Trinidad and Tobago', 'Ghana', 'North Macedonia',
    'Belarus', 'Togo', 'Rwanda', 'Libya', 'Guinea-Bissau', 'Burundi', 'Israel', 'Germany', 'Bangladesh', 'Nicaragua'
}

NAME_MAP = {
    'United States of America': 'United States', 'Republic of Korea': 'South Korea',
    'Russian Federation': 'Russia', 'Syrian Arab Republic': 'Syria', 'Türkiye': 'Turkey',
    'Viet Nam': 'Vietnam', "Côte d'Ivoire": 'Ivory Coast', 'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
}

ISO3_MAP = {
    'Greece': 'GRC', 'Zambia': 'ZMB', 'Italy': 'ITA', 'Thailand': 'THA', 'Canada': 'CAN', 'Peru': 'PER',
    'United States of America': 'USA', 'Dominican Republic': 'DOM', 'Republic of Korea': 'KOR', 'Kazakhstan': 'KAZ',
    'El Salvador': 'SLV', 'Suriname': 'SUR', 'Eswatini': 'SWZ', 'Oman': 'OMN', 'Austria': 'AUT', 'Kuwait': 'KWT',
    'Costa Rica': 'CRI', 'Argentina': 'ARG', 'Namibia': 'NAM', 'Chile': 'CHL', 'Spain': 'ESP', 'Panama': 'PAN',
    'Ecuador': 'ECU', 'Myanmar': 'MMR', 'Belgium': 'BEL', 'Qatar': 'QAT', 'Uruguay': 'URY', 'Guyana': 'GUY',
    'Morocco': 'MAR', 'Honduras': 'HND', 'Australia': 'AUS', 'Mauritania': 'MRT', 'Colombia': 'COL', 'France': 'FRA',
    'Japan': 'JPN', 'Haiti': 'HTI', 'New Zealand': 'NZL', 'Fiji': 'FJI', 'Niger': 'NER', 'Switzerland': 'CHE',
    'Poland': 'POL', 'Russian Federation': 'RUS', 'Somalia': 'SOM', 'Guatemala': 'GTM', 'Comoros': 'COM',
    'Mexico': 'MEX', 'Portugal': 'PRT', 'Armenia': 'ARM', 'Türkiye': 'TUR', 'Slovenia': 'SVN', 'China': 'CHN',
    'Syrian Arab Republic': 'SYR', 'Trinidad and Tobago': 'TTO', 'Ghana': 'GHA', 'North Macedonia': 'MKD',
    'Belarus': 'BLR', 'Togo': 'TGO', 'Rwanda': 'RWA', 'Libya': 'LBY', 'Guinea-Bissau': 'GNB', 'Burundi': 'BDI',
    'Israel': 'ISR', 'Germany': 'DEU', 'Bangladesh': 'BGD', 'Nicaragua': 'NIC',
}

def yield_to_kcal(df, yield_col):
    out = df.copy()
    out['kcal_per_ha'] = out[yield_col] * out['Item'].map(KCAL_PER_KG)
    return out

def optimal_crop_per_country(df):
    idx = df.groupby('Area')['kcal_per_ha'].idxmax()
    best = df.loc[idx, ['Area', 'Item', 'kcal_per_ha']].copy()
    best = best.rename(columns={'Item': 'optimal_crop'})
    return best[['Area', 'optimal_crop', 'kcal_per_ha']]

print('regression_df:', reg.shape)
print('predict:', pred.shape)

regression_df: (16388, 8)
predict: (936, 5)


In [10]:
y13 = reg[(reg['Year'].between(2011, 2013)) & (reg['Item'].isin(ITEMS))][['Area', 'Item', 'Yield']].copy()
y13 = y13[y13['Area'].isin(COUNTRY_65)]
y13 = y13.groupby(['Area', 'Item'], as_index=False)['Yield'].mean()
y13 = y13.rename(columns={'Yield': 'yield_kg'})
y13 = yield_to_kcal(y13, 'yield_kg')
opt_2013 = optimal_crop_per_country(y13)
opt_2013['scenario'] = '2013 (2011–2013平均)'

p = pred[pred['Area'].isin(COUNTRY_65)].copy()
p370m = p[p['scenario'] == 'SSP370'][['Area', 'Item', 'mean_yield']].rename(columns={'mean_yield': 'y'})
p370m = yield_to_kcal(p370m, 'y')
opt_370m = optimal_crop_per_country(p370m)
opt_370m['scenario'] = '2100 SSP370 平均'

p370c = p[p['scenario'] == 'SSP370'][['Area', 'Item', 'ci95_lower']].rename(columns={'ci95_lower': 'y'})
p370c = yield_to_kcal(p370c, 'y')
opt_370c = optimal_crop_per_country(p370c)
opt_370c['scenario'] = '2100 SSP370 95%CI下限'

p245m = p[p['scenario'] == 'SSP245'][['Area', 'Item', 'mean_yield']].rename(columns={'mean_yield': 'y'})
p245m = yield_to_kcal(p245m, 'y')
opt_245m = optimal_crop_per_country(p245m)
opt_245m['scenario'] = '2100 SSP245 平均'

p245c = p[p['scenario'] == 'SSP245'][['Area', 'Item', 'ci95_lower']].rename(columns={'ci95_lower': 'y'})
p245c = yield_to_kcal(p245c, 'y')
opt_245c = optimal_crop_per_country(p245c)
opt_245c['scenario'] = '2100 SSP245 95%CI下限'

all_opt = pd.concat([opt_2013, opt_370m, opt_370c, opt_245m, opt_245c], ignore_index=True)
all_opt['Country'] = all_opt['Area'].replace(NAME_MAP)
all_opt['ISO3'] = all_opt['Area'].map(ISO3_MAP)
order = ['2013 (2011–2013平均)', '2100 SSP370 平均', '2100 SSP370 95%CI下限', '2100 SSP245 平均', '2100 SSP245 95%CI下限']
all_opt['scenario'] = pd.Categorical(all_opt['scenario'], categories=order, ordered=True)

print('=== シナリオ別 国数 ===')
for s in order:
    n = (all_opt['scenario'] == s).sum()
    print(f'  {s}: {n}か国')
print()
print('=== シナリオ別 最適作物の内訳 ===')
print(all_opt.groupby(['scenario', 'optimal_crop'], observed=True).size().unstack(fill_value=0))
print(all_opt.head(10))

=== シナリオ別 国数 ===
  2013 (2011–2013平均): 65か国
  2100 SSP370 平均: 65か国
  2100 SSP370 95%CI下限: 65か国
  2100 SSP245 平均: 65か国
  2100 SSP245 95%CI下限: 65か国

=== シナリオ別 最適作物の内訳 ===
optimal_crop         Maize (corn)  Rice  Wheat
scenario                                      
2013 (2011–2013平均)             33    29      3
2100 SSP370 平均                 38    20      7
2100 SSP370 95%CI下限            38    23      4
2100 SSP245 平均                 37    21      7
2100 SSP245 95%CI下限            36    25      4
         Area  optimal_crop  kcal_per_ha            scenario     Country ISO3
0   Argentina  Maize (corn)   24668952.0  2013 (2011–2013平均)   Argentina  ARG
1     Armenia  Maize (corn)   24233352.0  2013 (2011–2013平均)     Armenia  ARM
2   Australia          Rice   29359718.4  2013 (2011–2013平均)   Australia  AUS
3     Austria  Maize (corn)   39757608.0  2013 (2011–2013平均)     Austria  AUT
4  Bangladesh  Maize (corn)   25515996.0  2013 (2011–2013平均)  Bangladesh  BGD
5     Belarus  Maize (corn)   2270

In [12]:
color_map = {'Maize (corn)': '#e74c3c', 'Rice': '#27ae60', 'Wheat': '#3498db'}
figs = []
for s in order:
    d = all_opt[all_opt['scenario'] == s].dropna(subset=['ISO3'])
    fig = px.choropleth(
        d,
        locations='ISO3',
        locationmode='ISO-3',
        color='optimal_crop',
        color_discrete_map=color_map,
        hover_name='Area',
        hover_data={'optimal_crop': True, 'kcal_per_ha': ':,.0f', 'ISO3': False},
        title=f'最適作物（カロリーベース） — {s}  ({len(d)}か国)'
    )
    fig.update_layout(
        margin=dict(l=0, r=0, t=50, b=0),
        height=420,
        geo=dict(showframe=False, showcoastlines=True),
        legend_title='最適作物'
    )
    fig.show()
    figs.append(fig)
print('5 maps done.')

5 maps done.


In [ ]:
out = all_opt[['scenario', 'Area', 'optimal_crop', 'kcal_per_ha']].copy()
out.to_csv(os.path.join(base_dir, 'optimal_crop_65countries_5scenarios.csv'), index=False)
print('Saved: optimal_crop_65countries_5scenarios.csv')